# CUDA + Python Practice
Run this notebook to verify your GPU is working and explore the basics.

## 1. Verify GPU is visible

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Thu Apr 23 21:29:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.02              Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5500 Laptop GPU    On  |   00000000:01:00.0 Off |                  Off |
| N/A   40C    P0             35W /  130W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. PyTorch GPU check

In [2]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU name: NVIDIA RTX A5500 Laptop GPU
CUDA version: 12.4


## 3. Simple GPU tensor operation with PyTorch

In [3]:
import torch

# Create tensors on GPU
a = torch.randn(1000, 1000, device='cuda')
b = torch.randn(1000, 1000, device='cuda')

# Matrix multiply on GPU
c = torch.matmul(a, b)
print('Result shape:', c.shape)
print('Result device:', c.device)

Result shape: torch.Size([1000, 1000])
Result device: cuda:0


## 4. CPU vs GPU speed comparison

In [4]:
import torch
import time

size = 5000

# CPU
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)
start = time.time()
c_cpu = torch.matmul(a_cpu, b_cpu)
cpu_time = time.time() - start
print(f'CPU: {cpu_time:.3f}s')

# GPU
a_gpu = a_cpu.cuda()
b_gpu = b_cpu.cuda()
torch.cuda.synchronize()  # wait for GPU to be ready
start = time.time()
c_gpu = torch.matmul(a_gpu, b_gpu)
torch.cuda.synchronize()  # wait for GPU to finish
gpu_time = time.time() - start
print(f'GPU: {gpu_time:.3f}s')
print(f'Speedup: {cpu_time / gpu_time:.1f}x')

CPU: 0.361s
GPU: 0.023s
Speedup: 15.6x


## 5. Write a custom CUDA kernel with Numba

In [5]:
from numba import cuda
import numpy as np

# Custom CUDA kernel: element-wise multiply
@cuda.jit
def multiply_kernel(a, b, out):
    idx = cuda.grid(1)  # global thread index
    if idx < a.size:
        out[idx] = a[idx] * b[idx]

n = 1_000_000
a = np.random.rand(n).astype(np.float32)
b = np.random.rand(n).astype(np.float32)
out = np.zeros(n, dtype=np.float32)

# Copy to GPU
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_out = cuda.to_device(out)

# Launch kernel: 256 threads per block
threads_per_block = 256
blocks = (n + threads_per_block - 1) // threads_per_block
multiply_kernel[blocks, threads_per_block](d_a, d_b, d_out)

result = d_out.copy_to_host()
expected = a * b
print('Max error:', np.max(np.abs(result - expected)))
print('First 5 results:', result[:5])

Max error: 0.0
First 5 results: [0.498232   0.04842509 0.08844438 0.00915993 0.17326713]


## 6. CuPy — NumPy but on the GPU

In [6]:
import cupy as cp
import numpy as np

# Works exactly like NumPy, runs on GPU
a = cp.random.rand(10000, 10000, dtype=cp.float32)
b = cp.random.rand(10000, 10000, dtype=cp.float32)

result = cp.dot(a, b)
print('Shape:', result.shape)
print('Device:', result.device)

# Convert back to NumPy if needed
result_np = cp.asnumpy(result[0, :5])
print('First 5 values:', result_np)

Shape: (10000, 10000)
Device: <CUDA Device 0>
First 5 values: [2495.019  2520.9553 2541.448  2496.1057 2523.0547]


In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)